In [0]:
from pyspark.sql import functions as F

In [0]:
#search access datalake storage
service_credential = dbutils.secrets.get(scope='hmdaScope',key='app-secret')

spark.conf.set("fs.azure.account.auth.type.hmdaprojstorage.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.hmdaprojstorage.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.hmdaprojstorage.dfs.core.windows.net", "df5e1b30-7a5d-46d4-a569-82706acec819")
spark.conf.set("fs.azure.account.oauth2.client.secret.hmdaprojstorage.dfs.core.windows.net", service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.hmdaprojstorage.dfs.core.windows.net", "https://login.microsoftonline.com/2ecf7998-4c57-4bc7-acbb-43cac8c31bfa/oauth2/token")

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS hmda")

spark.read.format("delta").load("abfss://silver@hmdaprojstorage.dfs.core.windows.net") \
    .write.format("delta").mode("overwrite").saveAsTable("hmda.silver_loans")

print("table registered")

table registered


In [0]:
%sql
-- 1. Loans and dollar volume by year
SELECT as_of_year,
       COUNT(*) AS num_loans,
       ROUND(SUM(loan_amount_000s)/1000, 0) AS total_millions
FROM hmda.silver_loans
WHERE action_taken_name = 'Loan originated'
GROUP BY as_of_year
ORDER BY as_of_year;

as_of_year,num_loans,total_millions
2013,7111460,1564612.0
2014,4822041,1114715.0
2015,6104778,1506028.0
2016,7028725,1807516.0
2017,5977417,1538378.0


In [0]:
%sql
-- 2. Top 10 states by loan volume
SELECT state_name, COUNT(*) AS num_loans, ROUND(AVG(loan_amount_000s),1) AS avg_amount
FROM hmda.silver_loans
WHERE action_taken_name = 'Loan originated'
GROUP BY state_name
ORDER BY num_loans DESC
LIMIT 10;

state_name,num_loans,avg_amount
California,4130012,399.3
Texas,2327060,213.3
Florida,1798653,216.9
Illinois,1238271,220.9
Ohio,1087023,156.3
Pennsylvania,1078520,185.1
Michigan,1026474,163.3
North Carolina,986926,201.1
Georgia,979947,205.5
Virginia,975267,289.4


In [0]:
%sql
-- 3. Approval rate by loan type
SELECT loan_type_name,
       COUNT(*) AS total,
       ROUND(100.0 * SUM(CASE WHEN action_taken_name='Loan originated' THEN 1 ELSE 0 END)/COUNT(*),1) AS approval_rate_pct
FROM hmda.silver_loans
GROUP BY loan_type_name
ORDER BY total DESC;

loan_type_name,total,approval_rate_pct
Conventional,22015480,100.0
FHA-insured,5402236,100.0
VA-guaranteed,2999474,100.0
FSA/RHS-guaranteed,627231,100.0


In [0]:
%sql
-- 4. Approval rate by applicant race (fair lending) — computed live from silver
SELECT applicant_race_name_1,
       COUNT(*) AS total,
       ROUND(100.0 * SUM(CASE WHEN action_taken_name='Loan originated' THEN 1 ELSE 0 END)/COUNT(*),1) AS approval_rate_pct
FROM hmda.silver_loans
WHERE action_taken_name IN ('Loan originated','Application denied by financial institution')
GROUP BY applicant_race_name_1
ORDER BY total DESC;

applicant_race_name_1,total,approval_rate_pct
White,24183565,100.0
"Information not provided by applicant in mail, Internet, or telephone application",3101922,100.0
Black or African American,1730137,100.0
Asian,1699376,100.0
American Indian or Alaska Native,183816,100.0
Native Hawaiian or Other Pacific Islander,124417,100.0
Not applicable,21188,100.0


In [0]:
%sql
-- 5. Top 10 lenders by loan volume (uses the joined panel data)
SELECT respondent_name,
       COUNT(*) AS num_loans,
       ROUND(SUM(loan_amount_000s)/1000, 0) AS total_millions
FROM hmda.silver_loans
WHERE action_taken_name = 'Loan originated'
  AND respondent_name IS NOT NULL
GROUP BY respondent_name
ORDER BY num_loans DESC
LIMIT 10;

respondent_name,num_loans,total_millions
WELLS FARGO BK NA,1696557,477119.0
QUICKEN LOANS INC.,1350969,276987.0
JPMORGAN CHASE BK NA,779169,240374.0
BANK OF AMER NA,715201,204398.0
U S BK NA,416693,99383.0
"QUICKEN LOANS, INC.",373394,77027.0
CITIBANK NA,360944,97985.0
FLAGSTAR BK FSB,355461,91323.0
FREEDOM MORTGAGE CORPORATION,346227,72448.0
LOANDEPOT.COM,298618,77367.0


In [0]:
%sql
-- 7. Year-over-year growth in originations
SELECT as_of_year,
       COUNT(*) AS num_loans,
       ROUND(100.0 * (COUNT(*) - LAG(COUNT(*)) OVER (ORDER BY as_of_year))
             / LAG(COUNT(*)) OVER (ORDER BY as_of_year), 1) AS yoy_growth_pct
FROM hmda.silver_loans
WHERE action_taken_name = 'Loan originated'
GROUP BY as_of_year
ORDER BY as_of_year;

as_of_year,num_loans,yoy_growth_pct
2013,7111460,null
2014,4822041,-32.2
2015,6104778,26.6
2016,7028725,15.1
2017,5977417,-15.0


In [0]:
%sql
-- 8. Approval rate by applicant sex
SELECT applicant_sex_name,
       COUNT(*) AS total,
       ROUND(100.0 * SUM(CASE WHEN action_taken_name='Loan originated' THEN 1 ELSE 0 END)/COUNT(*),1) AS approval_rate_pct
FROM hmda.silver_loans
WHERE action_taken_name IN ('Loan originated','Application denied by financial institution')
GROUP BY applicant_sex_name
ORDER BY total DESC;

applicant_sex_name,total,approval_rate_pct
Male,20567649,100.0
Female,8608269,100.0
"Information not provided by applicant in mail, Internet, or telephone application",1847220,100.0
Not applicable,21283,100.0


In [0]:
%sql
-- 9. Loan amount vs applicant income — affordability by state (top 15 by volume)
SELECT state_name,
       ROUND(AVG(loan_amount_000s),1) AS avg_loan,
       ROUND(AVG(applicant_income_000s),1) AS avg_income,
       ROUND(AVG(loan_amount_000s)/AVG(applicant_income_000s),2) AS loan_to_income_ratio
FROM hmda.silver_loans
WHERE action_taken_name = 'Loan originated'
  AND applicant_income_000s > 0
GROUP BY state_name
ORDER BY COUNT(*) DESC
LIMIT 15;

state_name,avg_loan,avg_income,loan_to_income_ratio
California,400.9,138.1,2.9
Texas,216.2,112.1,1.93
Florida,217.6,99.3,2.19
Illinois,221.5,113.3,1.96
Ohio,156.5,90.3,1.73
Pennsylvania,185.2,97.9,1.89
Michigan,163.9,89.0,1.84
New York,298.9,135.6,2.2
North Carolina,202.0,93.6,2.16
Georgia,208.3,97.0,2.15


In [0]:
%sql
-- 10. Conventional vs government-backed lending share by year
SELECT as_of_year,
       SUM(CASE WHEN loan_type_name='Conventional' THEN 1 ELSE 0 END) AS conventional,
       SUM(CASE WHEN loan_type_name<>'Conventional' THEN 1 ELSE 0 END) AS government_backed,
       ROUND(100.0 * SUM(CASE WHEN loan_type_name='Conventional' THEN 1 ELSE 0 END)/COUNT(*),1) AS conventional_pct
FROM hmda.silver_loans
WHERE action_taken_name = 'Loan originated'
GROUP BY as_of_year
ORDER BY as_of_year;

as_of_year,conventional,government_backed,conventional_pct
2013,5402814,1708646,76.0
2014,3454419,1367622,71.6
2015,4199640,1905138,68.8
2016,4850809,2177916,69.0
2017,4107798,1869619,68.7
